# Project 40 — Detect Mining Expansion

## 1. Introduce the Project

In this project, we will monitor the **expansion of an open-pit mining area between two years**.

The basic idea is:

> **Land that was not classified as mining before, but is classified as mining later = potential mining expansion.**

This is a simple but important form of **change detection**.

### Why does this matter?

Mining expansion can affect:

* vegetation
* soil
* water resources
* surrounding land cover
* habitat
* landscape structure

So satellite imagery can help us monitor the **spatial footprint of mining activity**.

---

# 2. Big Picture Workflow

```text
       Land-cover map: Year 1
                 │
                 ▼
          Extract mining
                 │
                 │
                 ▼
       Land-cover map: Year 2
                 │
                 ▼
          Extract mining
                 │
                 ▼
       Compare the two masks
                 │
                 ▼
        New mining areas
                 │
                 ▼
      Calculate expansion area
```

The important idea is:

```text
Mining in Year 2
        AND
Not mining in Year 1
        ↓
New mining area
```

---

# 3. Main Concept — Change Detection

We already used change detection in previous projects.

For example, in **Project 38**, we detected:

```text
Non-urban → Urban
```

In **Project 39**, we detected specific land-cover transitions such as:

```text
Tree cover → Cropland
Water → Cropland
```

Now we are doing something similar:

```text
Non-mining → Mining
```

So the logic is:

```javascript
var miningExpansion = miningLater
  .and(miningEarlier.not());
```

### Why `.not()`?

Suppose:

```text
miningEarlier = 1
```

means the pixel was mining.

Then:

```text
miningEarlier.not() = 0
```

We only want pixels that were **not mining before**.

So:

```text
Later mining = 1
Earlier non-mining = 1

1 AND 1 = 1
```

That pixel is classified as **new mining**.

---

# 4. Important Dataset Concept

For this lesson, we need a land-cover dataset containing a **mining/quarry class**.

We will use **ESA WorldCover**, but there is an important limitation:

**WorldCover does not provide a dedicated mining class.**

Its class:

```text
60 = Bare / sparse vegetation
```

can contain exposed mine surfaces, but it can also contain:

* natural bare soil
* deserts
* rocky surfaces
* other sparsely vegetated areas

Therefore, we should **not claim that every class-60 pixel is a mine**.

For this educational project, we will use a **simple mining study area where the mine is visually identifiable**, and demonstrate the change-detection workflow using a manually defined mining mask.

This is actually a useful lesson: **change detection is only as good as the classification/mask used as its input.**

---

# 5. Step-by-Step

## Step 1 — Define the mining area

We need a study area containing an open-pit mine.

We will use a small area around an example mine and work with Sentinel-2 imagery.

---

## Step 2 — Load Sentinel-2 imagery

We need imagery from two different periods.

```text
Earlier image
      ↓
Later image
      ↓
Compare
```

We will use:

```javascript
COPERNICUS/S2_SR_HARMONIZED
```

as our Sentinel-2 collection.

---

## Step 3 — Create a simple mining mask

For this lesson, we will identify exposed mining surfaces using a simple spectral approach.

Bare mining surfaces often have:

* relatively low vegetation
* high reflectance in some visible/SWIR bands
* little or no vegetation signal

A useful starting point is **NDVI**.

```text
Low NDVI
   ↓
little vegetation
   ↓
possible bare surface
   ↓
potential mining area
```

But remember:

> Low NDVI does **not** automatically mean mining.

It can also represent bare soil, rock, roads, construction sites, or deserts.

---

# 6. Important New Concept — Mining Mask

We can create a simple mask such as:

```javascript
var mining = ndvi.lt(0.2);
```

This means:

> Keep pixels where NDVI is below 0.2.

Then:

```javascript
selfMask()
```

allows us to display only those pixels.

For example:

```javascript
Map.addLayer(mining.selfMask(), {}, "Potential Mining");
```

The word **potential** is important.

We are not saying:

> "Every low-NDVI pixel is definitely a mine."

We are saying:

> "These pixels satisfy our simple criterion and may represent exposed mining surfaces."

---

# 7. Complete Code

```javascript
// ==========================================
// Project 40 — Detect Mining Expansion
// ==========================================

// 1. Define study area
// Example small area containing an open-pit mine
var studyArea = ee.Geometry.Polygon([
  [
    [34.85, 38.65],
    [35.05, 38.65],
    [35.05, 38.80],
    [34.85, 38.80],
    [34.85, 38.65]
  ]
]);

// ------------------------------------------
// 2. Load Sentinel-2 imagery for earlier year
// ------------------------------------------

var earlier = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
  .filterBounds(studyArea)
  .filterDate("2020-06-01", "2020-09-30")
  .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
  .median()
  .clip(studyArea);

// ------------------------------------------
// 3. Load Sentinel-2 imagery for later year
// ------------------------------------------

var later = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
  .filterBounds(studyArea)
  .filterDate("2025-06-01", "2025-09-30")
  .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
  .median()
  .clip(studyArea);

// ------------------------------------------
// 4. Calculate NDVI for both years
// ------------------------------------------

var ndviEarlier = earlier.normalizedDifference(["B8", "B4"]);

var ndviLater = later.normalizedDifference(["B8", "B4"]);

// ------------------------------------------
// 5. Create potential mining masks
// ------------------------------------------

// Low vegetation signal
var miningEarlier = ndviEarlier.lt(0.2);

var miningLater = ndviLater.lt(0.2);

// ------------------------------------------
// 6. Detect potential mining expansion
// ------------------------------------------

// Mining later AND not mining earlier
var miningExpansion = miningLater
  .and(miningEarlier.not());

// ------------------------------------------
// 7. Calculate expansion area
// ------------------------------------------

var expansionArea = ee.Image.pixelArea()
  .updateMask(miningExpansion)
  .reduceRegion({
    reducer: ee.Reducer.sum(),
    geometry: studyArea,
    scale: 10,
    maxPixels: 1e9
  });

// Convert m² to km²
var expansionKm2 = ee.Number(
  expansionArea.get("area")
).divide(1e6);

// ------------------------------------------
// 8. Print result
// ------------------------------------------

print(
  "Potential Mining Expansion (km²)",
  expansionKm2
);

// ------------------------------------------
// 9. Display imagery
// ------------------------------------------

Map.addLayer(
  earlier,
  {
    bands: ["B4", "B3", "B2"],
    min: 0,
    max: 3000
  },
  "Earlier Image"
);

Map.addLayer(
  later,
  {
    bands: ["B4", "B3", "B2"],
    min: 0,
    max: 3000
  },
  "Later Image"
);

// ------------------------------------------
// 10. Display potential mining
// ------------------------------------------

Map.addLayer(
  miningEarlier.selfMask(),
  {},
  "Potential Mining - Earlier"
);

Map.addLayer(
  miningLater.selfMask(),
  {},
  "Potential Mining - Later"
);

// ------------------------------------------
// 11. Display expansion
// ------------------------------------------

Map.addLayer(
  miningExpansion.selfMask(),
  {},
  "Potential Mining Expansion"
);

// ------------------------------------------
// 12. Center map
// ------------------------------------------

Map.centerObject(studyArea, 13);
```

---

# 8. Let's Understand the Most Important Lines

### Creating the mining mask

```javascript
var miningEarlier = ndviEarlier.lt(0.2);
```

`.lt()` means **less than**.

So:

```text
NDVI < 0.2
```

becomes:

```text
true  →  1
false →  0
```

---

### Detecting expansion

```javascript
var miningExpansion = miningLater
  .and(miningEarlier.not());
```

Read it as:

> "Find pixels that are mining later AND were not mining earlier."

This is the most important line of the project.

---

### Calculating area

```javascript
ee.Image.pixelArea()
```

gives the area of each pixel in square meters.

Then:

```javascript
.updateMask(miningExpansion)
```

keeps only the expansion pixels.

Finally:

```javascript
.reduceRegion({
  reducer: ee.Reducer.sum(),
  ...
})
```

adds those pixel areas together.

Then:

```javascript
.divide(1e6)
```

converts:

```text
m² → km²
```

---

# 9. Connect to Previous Lessons

This project combines several things you've already learned.

### Project 6 — NDVI

You learned:

```text
B8 + B4
   ↓
NDVI
```

Now we use NDVI to identify areas with little vegetation.

### Project 28 — Change Detection

You learned:

```text
Later − Earlier
```

and how to identify change.

Here we use a slightly different logical approach:

```text
Later mining
AND
Earlier NOT mining
```

### Project 38 — Urban Expansion

You learned:

```text
Urban later
AND
NOT urban earlier
```

The logic is almost identical:

```text
Project 38:
Non-urban → Urban

Project 40:
Non-mining → Mining
```

### Project 39 — Change Matrix

You learned how to identify:

```text
Class A → Class B
```

Mining expansion is essentially another land-cover transition:

```text
Non-mining → Mining
```

---

# 10. Environmental Impact Assessment

This is where the project becomes more interesting.

Detecting mining expansion is only the **first step**.

A real environmental assessment could continue:

```text
Mining expansion
       ↓
Vegetation loss
       ↓
NDVI change
       ↓
Water-body proximity
       ↓
Land-cover change
       ↓
Potential environmental impacts
```

For example, after detecting expansion we could ask:

> Did vegetation decrease around the newly expanded mine?

Or:

> Did the mine move closer to a river or lake?

Or:

> How much cropland was converted to mining?

These are much stronger environmental questions than simply measuring mine area.

---

# 11. Things to Remember

### 1. Change detection needs two time periods

```text
Earlier → Later
```

Without the earlier observation, we cannot determine expansion.

### 2. `AND` is extremely useful

```javascript
later.and(earlier.not())
```

means:

> Later condition is true, while earlier condition was false.

### 3. Low NDVI ≠ mining

This is extremely important.

Low NDVI can also mean:

```text
bare soil
rock
roads
urban surfaces
construction
desert
mining
```

So this project demonstrates the **change-detection workflow**, not a research-grade mining classifier.

### 4. Area is calculated from pixels

```text
pixelArea
     ↓
mask
     ↓
sum
     ↓
total area
```

### 5. Always think about what the classification actually means

A satellite pixel does not directly tell us:

> "I am a mine."

We need a reliable classification method to make that interpretation.

---

# 12. Common Beginner Mistakes

### Mistake 1 — Assuming all bare land is mining

```javascript
var mining = ndvi.lt(0.2);
```

does **not** mean:

> "This is definitely mining."

It means:

> "This has low NDVI."

---

### Mistake 2 — Using only the later image

If you only identify mining in 2025, you know where mining exists.

You **do not know whether it expanded**.

You need:

```text
2020 mining
+
2025 mining
```

---

### Mistake 3 — Reversing the logic

This:

```javascript
miningEarlier.and(miningLater.not())
```

would identify areas that were mining before but are **no longer classified as mining**.

That is closer to potential mining-area loss/reclassification, not expansion.

---

### Mistake 4 — Treating the threshold as universal

```javascript
ndvi.lt(0.2)
```

is an educational threshold.

A real study would test and validate an appropriate threshold for the specific environment and imagery.

---

# 13. Practice Project — Detect Mining Expansion Yourself

Now build your own version.

### Task 1 — Create the two mining masks

Create:

```text
miningEarlier
miningLater
```

using the NDVI approach.

---

### Task 2 — Detect expansion

Create:

```text
miningExpansion
```

using the logic:

```text
Later mining
AND
NOT earlier mining
```

---

### Task 3 — Calculate area

Calculate the expansion area in:

```text
km²
```

---

### Task 4 — Display four layers

Display:

1. Earlier image
2. Later image
3. Earlier potential mining
4. Mining expansion

---

### Task 5 — Environmental interpretation

Look at the expansion area and ask:

* Is it connected to the existing mine?
* Is it scattered or continuous?
* Does it overlap vegetation?
* Is it near water?
* Does it appear to represent actual mining expansion or could it be bare soil/rock?

**Do not assume that every detected pixel represents actual mining.**

---

# 14. Conceptual Questions

Answer these one at a time:

### Question 1

Suppose a pixel has:

```text
Earlier: NOT mining
Later:   mining
```

Should it be included in `miningExpansion`?

Explain **why**.

### Question 2

Why can't we simply use:

```javascript
ndviLater.subtract(ndviEarlier)
```

to determine that an area is definitely a mine?

### Question 3

What is the difference between:

```text
Low vegetation
```

and:

```text
Mining
```

from a Remote Sensing perspective?

### Question 4

Why is environmental impact assessment more than simply calculating the area of the mine?

---

## Core idea of Project 40

The most important pattern to remember is:

```text
             EARLIER             LATER
                │                   │
                ▼                   ▼
          Mining mask          Mining mask
                │                   │
                └─────────┬─────────┘
                          ▼
                 Later AND NOT Earlier
                          │
                          ▼
                 Mining Expansion
                          │
                          ▼
                    Area (km²)
```

This is the same change-detection thinking you've already developed in Projects **28, 30, 38, and 39**, now applied to a new environmental problem.
